In [4]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import keras
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import multilabel_confusion_matrix
from sklearn import metrics
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer
import matplotlib.pyplot as plt

import os

%matplotlib inline
plt.rcParams["figure.figsize"] = (10,6)

## NLTK resources

In [5]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [6]:
# I just directly downloaded the files from the provided URLs and read them into pandas DataFrames.
train_path = keras.utils.get_file(
    "train.csv.zip", 
    "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/train.csv.zip"
)
train_df = pd.read_csv(train_path)

test_path = keras.utils.get_file(
    "test.csv.zip", 
    "https://jrssbcrsefilesnait.blob.core.windows.net/3950data1/test.csv.zip"
)
test_df = pd.read_csv(test_path)

## Project 1 - NLP and Text Classification

For this project you will need to classify some angry comments into their respective category of angry. The process that you'll need to follow is (roughly):
<ol>
<li> Use NLP techniques to process the training data. 
<li> Train model(s) to predict which class(es) each comment is in.
    <ul>
    <li> A comment can belong to any number of classes, including none. 
    </ul>
<li> Generate predictions for each of the comments in the test data. 
<li> Write your test data predicitions to a CSV file, which will be scored. 
</ol>

You can use any models and NLP libraries you'd like. 

# **Raphael's Model** 

## Tokenizer

In [9]:
stop_words = stopwords.words('english')

class lemmaTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
        self.wnl = WordNetLemmatizer()
    def __call__(self, doc):
        doc = re.sub('[^a-zA-Z]', ' ', doc)
        tokens = nltk.word_tokenize(doc.lower())
        return [self.wnl.lemmatize(t) for t in tokens if t not in self.stop_words]

## Pipeline and Targets

In [72]:
categories = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

pipeline = Pipeline([
                    ("vect", TfidfVectorizer(tokenizer=lemmaTokenizer(stop_words), 
                         max_features=5000, 
                         ngram_range=(1,2))),
                    ("model", LogisticRegression(C=2.0, class_weight='balanced', solver='liblinear'))
])

## Train and Predict

In [73]:
predictions = {'id': test_df['id']}

for category in categories:
    pipeline.fit(train_df['comment_text'], train_df[category])
    predictions[category] = pipeline.predict(test_df['comment_text'])

c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Chabiloo\anaconda3\Li

## CSV file output

In [ ]:
predictions.to_csv('output/out1.csv', index=False) 

### Originally ran an F1 score check for it and average for the six categories was just 49%. Deleted it because it was taking too long and we needed to submit on time.

# **Ateeb's Model**

## NLP Processing and Model Training

In [27]:
# Download the list of common English stopwords (e.g., "the", "is", "and")
# These are used to remove unimportant words during text preprocessing
nltk.download('stopwords')

# Download the WordNet lexical database used for lemmatization
# Lemmatization reduces words to their base form (e.g., "running" -> "run")
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Text Preprocessing

In [28]:
# Load English stopwords
stop_words = set(stopwords.words('english'))

# Create lemmatizer object
lemmatizer = WordNetLemmatizer()

# Function to clean text
def clean_text(text):

    text = text.lower()  # convert text to lowercase

    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove numbers and punctuation

    words = text.split()  # split text into words

    # remove stopwords and apply lemmatization
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(words)  # join words back into sentence

In [29]:
# Apply text cleaning function to training and test comments
train_df["clean_text"] = train_df["comment_text"].apply(clean_text)
test_df["clean_text"] = test_df["comment_text"].apply(clean_text)

### Feature Extraction using TF-IDF

In [30]:
# Convert text into TF-IDF numerical features
vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1,2),   # use unigrams and bigrams
    stop_words='english',
    sublinear_tf=True,
    min_df=2
)

# Fit on training data and transform it
X_train = vectorizer.fit_transform(train_df["clean_text"])

# Transform test data using the same vectorizer
X_test = vectorizer.transform(test_df["clean_text"])

### Model Training

In [31]:
# Select the target labels for multi-label classification
y_train = train_df[[
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]]

In [32]:
# Split training data into training and validation sets (80/20)
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

### Train Model

In [33]:
# Initialize One-vs-Rest classifier with LinearSVC
model = OneVsRestClassifier(LinearSVC())

# Train the model on the training split
model.fit(X_train_split, y_train_split)

OneVsRestClassifier(estimator=LinearSVC())

### Calculate Accuracy

In [34]:
# Predict validation data
val_predictions = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, val_predictions)

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.918439605201316


In [35]:
# Define the output columns
columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

print("Classification Report:", classification_report(y_val, val_predictions,target_names=columns))

Classification Report:                precision    recall  f1-score   support

        toxic       0.86      0.69      0.76      3056
 severe_toxic       0.50      0.25      0.33       321
      obscene       0.89      0.69      0.78      1715
       threat       0.60      0.20      0.30        74
       insult       0.79      0.57      0.66      1614
identity_hate       0.74      0.28      0.40       294

    micro avg       0.84      0.62      0.71      7074
    macro avg       0.73      0.45      0.54      7074
 weighted avg       0.83      0.62      0.70      7074
  samples avg       0.06      0.06      0.06      7074



c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Prediction on Test Data

In [36]:
# Predict labels for the test data
predictions = model.predict(X_test)

# Convert predictions to a DataFrame
pred_df = pd.DataFrame(predictions, columns=columns)

# Add the 'id' column from the test set
pred_df.insert(0, "id", test_df["id"])

In [38]:
# Save predictions to CSV for submission
pred_df.to_csv("output/out2.csv", index=False)

# Display first few rows of predictions
pred_df.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,1,0,1,0,1,0
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0


### Model & Approach Description

- **Model:** LinearSVC with One-vs-Rest strategy for multi-label classification.  
- **Text Processing:** Lowercased text, removed non-letter characters, removed stopwords, and applied lemmatization.  
- **Features:** TF-IDF vectorization with unigrams and bigrams.  
- **Validation:** Training data split 80/20 for validation.  
- **Validation Accuracy:** 0.918 (~91.8%).  
- **weighted F1-score of 0.70**, indicating good overall performance.  
- Performance is higher for common classes such as *toxic* and *obscene*, while rarer classes like *threat* have lower recall due to limited training samples.
- **Output:** Predictions for test data saved to `output/out.csv`.

# **Marci's Model**

##### ***What type of vectorization should we use?***
- CountVectorizer()
- TfidfVectorizer()
- Word2Vec()

##### ***What strategies should we try?***
- Stemming
- Lemming
- Dimensionality reduction
    - SVD
    - Truncated SVD
    - PCA

##### ***How to classify?***
- RandomForestClassifier()

# Get started

In [40]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
 8   clean_text     159571 non-null  object
dtypes: int64(6), object(3)
memory usage: 11.0+ MB


In [42]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153164 entries, 0 to 153163
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            153164 non-null  int64 
 1   comment_text  153164 non-null  object
 2   clean_text    153164 non-null  object
dtypes: int64(1), object(2)
memory usage: 3.5+ MB


In [43]:
# Basic cleaning

train_df['comment_text'] = (
    train_df['comment_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip())

test_df['comment_text'] = (
    test_df['comment_text']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip())


In [44]:
train_df.sample(3)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,clean_text
83036,de259ff15784e8f9,fuck you i'll fuck you muslim turkish animal. ...,1,1,1,0,1,1,fuck fuck muslim turkish animal
103463,29a1cb230c00e3a0,""" bridge number? at here, you added the bridge...",0,0,0,0,0,0,bridge number added bridge number data file ac...
43782,74db87a47498ee81,""" this is very non-notable as related to the a...",0,0,0,0,0,0,non notable related archetype whole yes dragon...


In [45]:
test_df.sample(3)

,id,comment_text,clean_text
81447,81448,"==arbitration for == hi, you don’t know me but...",arbitration hi know contact mutual person got ...
20611,20612,:::::that map doesn't give a source though doe...,map give source though need want go ga contain...
69722,69723,":what can i say, i got my first influence to e...",say got first influence edit guy cut paste kno...


## **Models**

In [46]:
# Split X and y

target = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

y = train_df[target]
X = train_df['comment_text']
X_train, X_test, y_train, y_test = train_test_split(X, y)

### *Model 1: Basic*

In [47]:
# Build pipeline

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', OneVsRestClassifier(LinearSVC()))
    ])

pipe.fit(X_train, y_train)

# Make predictions
y_pred = pipe.predict(X_test)

In [48]:
# Evaluate model

print(classification_report(y_test, y_pred, target_names=target))

               precision    recall  f1-score   support

        toxic       0.88      0.69      0.77      3759
 severe_toxic       0.50      0.29      0.37       394
      obscene       0.88      0.69      0.77      2102
       threat       0.67      0.33      0.45       117
       insult       0.80      0.62      0.70      1933
identity_hate       0.70      0.27      0.39       361

    micro avg       0.84      0.63      0.72      8666
    macro avg       0.74      0.48      0.57      8666
 weighted avg       0.83      0.63      0.72      8666
  samples avg       0.06      0.06      0.06      8666



c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [49]:
multilabel_confusion_matrix(y_test, y_pred)

array([[[35782,   352],
        [ 1175,  2584]],

       [[39385,   114],
        [  278,   116]],

       [[37591,   200],
        [  649,  1453]],

       [[39757,    19],
        [   78,    39]],

       [[37652,   308],
        [  735,  1198]],

       [[39491,    41],
        [  265,    96]]])

#### *Model 1 Performance:*
- Classification report:
    - * Total data points only 8,866.  Really thought there'd be more.  Pretty sure that can't be correct.
    - toxic:
        - Lots of data points so metrics can be relied on
        - Few false positives, some false negatives, f1-score is not bad but could be better
        - Model did a decent job of classifying
    - severe_toxic:
        - Few data points
        - Scores are poor
        - Model did a poor job of classifying
    - obscene:
        - Lots of data points so metrics can be relied on
        - Few false positives, some false negatives, f1-score is not bad but could be better
        - Model did a decent job of classifying
    - threat:
        - Very few data points, metrics not reliable
        - Not many false positives, a lot of false negatives, but again - not reliable
        - Model did a poor job of classifying
    - insult:
        - Fair number of data points
        - OK f1-score, definetly could be better
    - identity_hate:
        - Few data points
        - False positive score not horrible, but false neg is bad - not reliable
    - micro/macro average ok-ish
    - weighted averages ok but I would like them to be higher and more balanced
    - samples avg is super low
- Multi-class label matrix
    - True negatives really hight
    - True positives really low
    - False negatives kind of high in some areas
- ***Overall, I don't think this model was terrible, but it also wasn't great at predicting classes***

#### Create some helper functions

In [50]:
for package in ['stopwords','punkt','wordnet']:
    nltk.download(package)

stop_words = set(stopwords.words('english')) 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Chabiloo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [51]:
# Custom stop word tokenizer

class swTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                filtered_tok.append(tok)
        return filtered_tok

In [52]:
# Custom stem word tokenizer

class stemTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
        from nltk.stem import SnowballStemmer
        self.stemmer = SnowballStemmer(language='english')
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                filtered_tok.append(self.stemmer.stem(tok))
        return filtered_tok

In [53]:
# Custom lemmatization tokenizer

class lemmaTokenizer(object):
    def __init__(self, stop_words):
        self.stop_words = stop_words
        from nltk.stem import WordNetLemmatizer
        self.lemmatizer = WordNetLemmatizer()
    def __call__(self, doc):
        tokens = word_tokenize(doc)
        filtered_tok = []
        for tok in tokens:
            if tok not in stop_words:
                filtered_tok.append(self.lemmatizer.lemmatize(tok))
        return filtered_tok

In [54]:
# Integrate rare tokenizers

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))        # Load a list of common stop words and convert to a set (cuz it's faster, I think)

lemma_tok = lemmaTokenizer(stop_words)              # Create an instance custom lemmaTokenizer class

In [55]:
# Custom function to clean text

def clean_toxic_text(text):

    # Lowercase everything
    text = text.lower()

# Normalize obfuscated characters (cleans up altered offensive words ex. 'st00pid')
    substitutions = {
        '@': 'a',
        '1': 'i',
        '!': 'i',
        '3': 'e',
        '4': 'a',
        '0': 'o',
        '$': 's',
        '7': 't'
    }
    for k, v in substitutions.items():
        text = text.replace(k, v)                                                           # Replace text found in keys with cooresponding value

    # Remove repeated characters (ex. 'stuuuuupid' = 'stuupid')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)                                              # Find characters repeated 3 or more times and reduce to 2

    # Remove spacing tricks ('i d i o t' = 'idiot')
    text = re.sub(r'(\b\w\s+){2,}\w\b', lambda m: m.group(0).replace(' ', ''), text)        # Detects letter, space, letter, space patterns and collapses them

    # Remove non-alphanumeric (except punctuation)
    text = re.sub(r"[^a-z0-9!?.,']", ' ', text)                                             # Removes emojis, symbols, weird punctuation (keeps normal stuff like !?,.')

    # Fix whitespace
    text = re.sub(r'\s+', ' ', text).strip()                                                # Removes extra spaces like double spaces, tabs, newlines, and trims ends

    return text

In [56]:
# I don't really get this, but apparently I need it...

from sklearn.base import BaseEstimator, TransformerMixin

class TextCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, clean_func):
        self.clean_func = clean_func

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.apply(self.clean_func)

In [57]:
# Custom function to oversample rare labels

from sklearn.utils import resample

def oversample_rare_labels(df, rare_labels, factor=3):
    rare = df[df[rare_labels].sum(axis=1) > 0]                      # Select rare rows - count rare labels per row and select rows with count greater than 0
    common = df[df[rare_labels].sum(axis=1) == 0]                   # Select common rows - count rare labels per row and select rows with count equal to 0

    rare_oversampled = resample(
        rare,
        replace=True,                                               # Allow sampling more than once
        n_samples=len(rare) * factor,                               # Increase rows by chosen factor 
        random_state=42
    )

    return pd.concat([common, rare_oversampled], axis=0).sample(frac=1, random_state=42)       # frac=1 means use all of the data in the sample (basically just shuffling the data)

### *Model 2:*
- extra cleaning
- tokenization
- oversampling of rare data
- set a few hyperparameters

In [58]:
# Perform oversampling

# Combine the training data
train_data = pd.concat([X_train, y_train], axis=1)

# Call the function
train_df_os = oversample_rare_labels(train_data, rare_labels=['threat','identity_hate','severe_toxic'])

# Resplit train_data
X_train_os = train_df_os['comment_text']
y_train_os = train_df_os[target]

In [59]:
# Build the pipeline

pipe = Pipeline([
    ('clean', TextCleaner(clean_toxic_text)),                               # Use custom clean function
    ('tfidf', TfidfVectorizer(
        tokenizer=lemma_tok,                                                # Use custom tokenizer (lemmatization)
        ngram_range=(1,2),                                                  # look for 1 and 2 word tokens
        min_df=3,                                                           # Ignore terms that appear less than 3 times
        max_features=100000                                                 # Vocab dictionary capped at this number
    )),
    ('clf', OneVsRestClassifier(LinearSVC()))    
])


In [60]:
# Fit pipeline to training data
pipe.fit(X_train_os, y_train_os)

c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('clean',
                 TextCleaner(clean_func=<function clean_toxic_text at 0x000001F282B67920>)),
                ('tfidf',
                 TfidfVectorizer(max_features=100000, min_df=3,
                                 ngram_range=(1, 2),
                                 tokenizer=<__main__.lemmaTokenizer object at 0x000001F280FF30E0>)),
                ('clf', OneVsRestClassifier(estimator=LinearSVC()))])

In [61]:
# Make predictions
y_pred = pipe.predict(X_test)

In [62]:
# Evaluate model

print(classification_report(y_test, y_pred, target_names=target))

               precision    recall  f1-score   support

        toxic       0.86      0.68      0.76      3759
 severe_toxic       0.42      0.30      0.35       394
      obscene       0.86      0.69      0.76      2102
       threat       0.64      0.32      0.43       117
       insult       0.77      0.59      0.67      1933
identity_hate       0.56      0.29      0.39       361

    micro avg       0.81      0.62      0.71      8666
    macro avg       0.68      0.48      0.56      8666
 weighted avg       0.80      0.62      0.70      8666
  samples avg       0.06      0.05      0.05      8666



c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


- Model 1 results
    - ![image.png](attachment:image.png)

In [63]:
# Evaluate model

multilabel_confusion_matrix(y_test, y_pred)

array([[[35702,   432],
        [ 1195,  2564]],

       [[39332,   167],
        [  274,   120]],

       [[37563,   228],
        [  659,  1443]],

       [[39755,    21],
        [   79,    38]],

       [[37611,   349],
        [  788,  1145]],

       [[39449,    83],
        [  255,   106]]])

#### *Model 2 Performance:*
- Classification report:
    - *Well that's annoying. It doesn't really look any better than the basic model...*
        - Data points decreased, but I think 8600 is actually a reasonable number
        - False positives increased slightly across the board
        - False negatives stayed more or less the same
        - f1-score jumped around a bit but increased just a tiny bit overall
- ***The oversampling didn't seem to help the recall scores***
- ***Tuning and extra cleaning might have made a small difference in precision scores***

## **Marci's final predictions on test_df**

In [64]:
# Retrain model with 100% of data

pipe.fit(train_df['comment_text'], train_df[target])

c:\Users\Chabiloo\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('clean',
                 TextCleaner(clean_func=<function clean_toxic_text at 0x000001F282B67920>)),
                ('tfidf',
                 TfidfVectorizer(max_features=100000, min_df=3,
                                 ngram_range=(1, 2),
                                 tokenizer=<__main__.lemmaTokenizer object at 0x000001F280FF30E0>)),
                ('clf', OneVsRestClassifier(estimator=LinearSVC()))])

In [65]:
# Make predictions

y_pred = pipe.predict(test_df['comment_text'])

In [66]:
# Convert array to df

pred_df = pd.DataFrame(y_pred, columns=target)

In [67]:
# Attach 'id' column (reset index)

final_df = pd.concat([
    test_df['id'].reset_index(drop=True), 
    pred_df.reset_index(drop=True)], 
    axis=1)

In [68]:
final_df.sample(5)

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
145612,145613,0,0,0,0,0,0
7597,7598,0,0,0,0,0,0
40942,40943,1,0,0,0,1,0
96073,96074,0,0,0,0,0,0
121881,121882,1,0,0,0,0,0


In [69]:
# Make sure everything lines up

len(final_df) == len(test_df)

True

In [70]:
# Check for nulls

test_df['comment_text'].isna().sum()

np.int64(0)

In [71]:
# Export predictions to CSV

final_df.to_csv('output/out3.csv', index=False)  

# Marci's second model was chosen. Better accuracy and more data points. Her csv output was submitted.